In [0]:
from pyspark.sql import functions as F

In [0]:
# ============================================================
# BRONZE NOTEBOOK PARAMETERS
# ============================================================

dbutils.widgets.text("catalog", "sales")
dbutils.widgets.text("raw_schema", "raw_layer")
dbutils.widgets.text("bronze_schema", "bronze")

catalog = dbutils.widgets.get("catalog")
raw_schema = dbutils.widgets.get("raw_schema")
bronze_schema = dbutils.widgets.get("bronze_schema")

print(f"Catalog       : {catalog}")
print(f"Raw Schema    : {raw_schema}")
print(f"Bronze Schema : {bronze_schema}")

In [0]:
# ============================================================
# CONFIGURATION
# ============================================================

RAW_BASE_PATH = (
    f"/Volumes/{catalog}/{raw_schema}/sales_generated_data"
)

BRONZE_TABLE = (
    f"{catalog}.{bronze_schema}.sales"
)

SOURCE_SYSTEMS = [
    "pos",
    "e_commerce",
    "mobile_app",
    "retail_store",
    "marketplace"
]

print(f"RAW PATH      : {RAW_BASE_PATH}")
print(f"BRONZE TABLE  : {BRONZE_TABLE}")
print(f"SOURCES       : {SOURCE_SYSTEMS}")

In [0]:
# ============================================================
# CREATE BRONZE SCHEMA
# ============================================================

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {catalog}.{bronze_schema}
""")

In [0]:
# ============================================================
# READ ALL SOURCE JSON FILES
# ============================================================

source_paths = [
    f"{RAW_BASE_PATH}/{source}/sales.json"
    for source in SOURCE_SYSTEMS
]

print("Source files:")
for path in source_paths:
    print(path)

In [0]:
# ============================================================
# READ JSON DATA
# ============================================================

raw_df = (
    spark.read
    .json(source_paths)
)

print(f"Total records read: {raw_df.count()}")

In [0]:
%skip
display(raw_df)

In [0]:
# ============================================================
# ADD BRONZE METADATA
# ============================================================

bronze_df = (
    raw_df

    # File from which the record was ingested
    .withColumn(
        "_source_file",
        F.col("_metadata.file_path")
    )

    # Bronze ingestion timestamp
    .withColumn(
        "_bronze_ingestion_timestamp",
        F.current_timestamp()
    )

    # Date on which Bronze ingestion happened
    .withColumn(
        "_bronze_ingestion_date",
        F.current_date()
    )

    # Generate hash for record-level lineage
    .withColumn(
        "_record_hash",
        F.sha2(
            F.to_json(
                F.struct(*raw_df.columns)
            ),
            256
        )
    )
)

In [0]:
# ============================================================
# BASIC BRONZE VALIDATION
# ============================================================

print("Record count:")
print(bronze_df.count())

print("\nColumns:")
bronze_df.printSchema()

print("\nSource distribution:")
(
    bronze_df
    .groupBy("data_source")
    .count()
    .orderBy("data_source")
    .show()
)

In [0]:
# ============================================================
# WRITE BRONZE TABLE
# ============================================================

(
    bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

print(f"Bronze table created successfully: {BRONZE_TABLE}")

In [0]:
# ============================================================
# VERIFY BRONZE TABLE
# ============================================================

spark.sql(f"""
SELECT COUNT(*) AS total_records
FROM {BRONZE_TABLE}
""").show()